# C1_01_Baselines
- X = X_flat_c1_v2.npy
- Y = y_labels_c1_v2.csv
- Built by Garmin_C1_DataBuild



## 1. Setup and configuration

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
np.random.seed(42)

DATA_DIR = Path(r"C:\dev\UoW\DATAX504\Final Assignment\garmin_data\Processed")

X_FLAT_PATH = DATA_DIR / "X_flat_c1_v2.npy"
Y_LABELS_PATH = DATA_DIR / "y_labels_c1_v2.csv"


## 2. Load X and Y

In [2]:
X_flat = np.load(X_FLAT_PATH)
y_labels_df = pd.read_csv(Y_LABELS_PATH, parse_dates=["date"])
print(f"Loaded X_flat {X_flat.shape} and y_labels_df {len(y_labels_df)} rows")
print(y_labels_df["c1_target"].value_counts())

Loaded X_flat (197341, 121) and y_labels_df 197341 rows
c1_target
Base         76998
Declining    62930
Improving    53674
Recovery      3739
Name: count, dtype: int64


## 3. Split by run, encode target

In [18]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_labels_df["c1_target"])      # class names to numbers
print("Classes:", dict(zip(le.classes_, range(len(le.classes_)))))

run_idx_all = y_labels_df["run_idx"].to_numpy()             # the run number of every window
unique_runs = np.unique(run_idx_all)
print("run_idx_all (first 5):", run_idx_all[:5], "| windows:", len(run_idx_all))

# Test runs are set aside (same 80/20 split as every other C1 notebook, for
# later use once a final model is chosen) but TEST DATA IS NOT BUILT OR USED
# ANYWHERE BELOW in this notebook — only trainval_runs is touched further.
# split by run: all windows of a run stay in one set
trainval_runs, test_runs = train_test_split(unique_runs, test_size=0.2, random_state=42)  # 20% for test
trainval_mask = np.isin(run_idx_all, trainval_runs)    # True for each window that belongs to a train or validation run
print("unique_runs:", len(unique_runs), "| trainval_runs:", len(trainval_runs), "| test_runs:", len(test_runs))
print("trainval_mask (first 5):", trainval_mask[:5], "| True count:", trainval_mask.sum())

X_trainval_raw = X_flat[trainval_mask]     # keeps the rows of X where the mask is True
y_trainval = y_encoded[trainval_mask]      # keeps the matching labels

# Validation split, carved from the trainval runs only.
trainval_run_idx = run_idx_all[trainval_mask]
trainval_unique_runs = np.unique(trainval_run_idx)
print("trainval_run_idx (first 5):", trainval_run_idx[:5], "| windows:", len(trainval_run_idx))
train_runs, val_runs = train_test_split(trainval_unique_runs, test_size=0.15, random_state=42)  # 15% for validation
train_mask = np.isin(trainval_run_idx, train_runs)
val_mask = np.isin(trainval_run_idx, val_runs)
print("train_runs:", len(train_runs), "| val_runs:", len(val_runs))
print("train_mask True:", train_mask.sum(), "| val_mask True:", val_mask.sum())

X_train_raw, X_val_raw = X_trainval_raw[train_mask], X_trainval_raw[val_mask]    # train windows and validation windows
y_train, y_val = y_trainval[train_mask], y_trainval[val_mask]

print(f"Train: {len(y_train)}, Val: {len(y_val)} (test set intentionally not built in this notebook)")

Classes: {'Base': 0, 'Declining': 1, 'Improving': 2, 'Recovery': 3}
run_idx_all (first 5): [149 149 149 149 149] | windows: 197341
unique_runs: 1875 | trainval_runs: 1500 | test_runs: 375
trainval_mask (first 5): [ True  True  True  True  True] | True count: 157511
trainval_run_idx (first 5): [149 149 149 149 149] | windows: 157511
train_runs: 1275 | val_runs: 225
train_mask True: 132966 | val_mask True: 24545
Train: 132966, Val: 24545 (test set intentionally not built in this notebook)


## 4. Naive baseline

In [16]:
naive_pred = [Counter(y_train).most_common(1)[0][0]] * len(y_val)
naive_acc = accuracy_score(y_val, naive_pred)
print(f"Naive validation accuracy: {naive_acc:.3f}")
print("\nNaive classification report (validation):")
print(classification_report(y_val, naive_pred, labels=np.arange(len(le.classes_)), target_names=le.classes_, zero_division=0))


Naive validation accuracy: 0.397

Naive classification report (validation):
              precision    recall  f1-score   support

        Base       0.40      1.00      0.57      9739
   Declining       0.00      0.00      0.00      7733
   Improving       0.00      0.00      0.00      6585
    Recovery       0.00      0.00      0.00       488

    accuracy                           0.40     24545
   macro avg       0.10      0.25      0.14     24545
weighted avg       0.16      0.40      0.23     24545



## 5. Random Forest

In [5]:
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train_raw, y_train)
rf_pred = rf.predict(X_val_raw)
rf_acc = accuracy_score(y_val, rf_pred)
print(f"Random Forest validation accuracy: {rf_acc:.3f}")
print("\nRandom Forest classification report (validation):")
print(classification_report(y_val, rf_pred, labels=np.arange(len(le.classes_)), target_names=le.classes_, zero_division=0))

print("\nFeature importance (grouped by field):")
feature_names = (["hr"] * 30) + (["speed"] * 30) + (["cadence"] * 30) + (["temperature"] * 30) + ["distance"]
importance_by_field = pd.Series(rf.feature_importances_, index=feature_names).groupby(level=0).sum()
print(importance_by_field.sort_values(ascending=False))


Random Forest validation accuracy: 0.567

Random Forest classification report (validation):
              precision    recall  f1-score   support

        Base       0.56      0.54      0.55      9739
   Declining       0.57      0.58      0.58      7733
   Improving       0.55      0.58      0.56      6585
    Recovery       0.77      0.84      0.80       488

    accuracy                           0.57     24545
   macro avg       0.61      0.63      0.62     24545
weighted avg       0.57      0.57      0.57     24545


Feature importance (grouped by field):
cadence        0.396364
speed          0.303782
hr             0.217996
temperature    0.067896
distance       0.013962
dtype: float64


## 6. Simple DNN

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

n_classes = len(le.classes_)
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(n_classes, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100, batch_size=64, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
)
dnn_pred = model.predict(X_val_scaled, verbose=0).argmax(axis=1)
dnn_acc = accuracy_score(y_val, dnn_pred)
print(f"Simple DNN validation accuracy: {dnn_acc:.3f}")
print("\nSimple DNN classification report (validation):")
print(classification_report(y_val, dnn_pred, labels=np.arange(len(le.classes_)), target_names=le.classes_, zero_division=0))


Epoch 1/100
2078/2078 - 17s - 8ms/step - accuracy: 0.5006 - loss: 1.0344 - val_accuracy: 0.5505 - val_loss: 0.9688
Epoch 2/100
2078/2078 - 11s - 5ms/step - accuracy: 0.5414 - loss: 0.9795 - val_accuracy: 0.5624 - val_loss: 0.9538
Epoch 3/100
2078/2078 - 20s - 10ms/step - accuracy: 0.5482 - loss: 0.9679 - val_accuracy: 0.5649 - val_loss: 0.9455
Epoch 4/100
2078/2078 - 11s - 5ms/step - accuracy: 0.5516 - loss: 0.9624 - val_accuracy: 0.5678 - val_loss: 0.9402
Epoch 5/100
2078/2078 - 10s - 5ms/step - accuracy: 0.5535 - loss: 0.9587 - val_accuracy: 0.5695 - val_loss: 0.9344
Epoch 6/100
2078/2078 - 11s - 5ms/step - accuracy: 0.5557 - loss: 0.9557 - val_accuracy: 0.5729 - val_loss: 0.9301
Epoch 7/100
2078/2078 - 11s - 5ms/step - accuracy: 0.5558 - loss: 0.9531 - val_accuracy: 0.5708 - val_loss: 0.9284
Epoch 8/100
2078/2078 - 10s - 5ms/step - accuracy: 0.5580 - loss: 0.9521 - val_accuracy: 0.5718 - val_loss: 0.9283
Epoch 9/100
2078/2078 - 11s - 5ms/step - accuracy: 0.5593 - loss: 0.9497 - val_

## 7. Summary (VALIDATION accuracy; test set untouched)


In [7]:
print(f"{'Model':<20}{'Accuracy':<12}")
print(f"{'Naive':<20}{naive_acc:<12.3f}")
print(f"{'Random Forest':<20}{rf_acc:<12.3f}")
print(f"{'Simple DNN':<20}{dnn_acc:<12.3f}")


Model               Accuracy    
Naive               0.397       
Random Forest       0.567       
Simple DNN          0.575       
